In [1]:
import sys
from pathlib import Path

import numpy as np


def find_repo_root(start_dir: Path) -> Path:
    """Locate the repository root so local package imports work in notebooks.

    Parameters:
        start_dir: Current working directory used by the notebook kernel.
    Returns:
        Path to the repo root containing both `balatro_gym` and `cs590_env`.
    """
    for candidate in (start_dir, *start_dir.parents):
        if (candidate / "balatro_gym").is_dir() and (candidate / "cs590_env").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook working directory.")


# VS Code/Jupyter kernels do not always start in the repo root.
repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from balatro_gym.balatro_env_2 import BalatroEnv
from cs590_env import BalatroPhaseWrapper, WrapperAction

# Create the wrapped environment and get the first observation.
env = BalatroPhaseWrapper(BalatroEnv(seed=0))
obs, info = env.reset()

# hand_levels rows are [id, level, chip, mult] in HandType enum order.
hand_levels = obs["hand_levels"]
high_card_id, high_card_level, high_card_chip, high_card_mult = hand_levels[0]
print("high_card [id, level, chip, mult]:", [
    int(high_card_id),
    int(high_card_level),
    int(high_card_chip),
    int(high_card_mult),
])


In [2]:

# Move from TRANSITION to COMBAT by selecting the current blind.
blind_action = int(WrapperAction.SELECT_BLIND_BASE + (int(obs["round"]) - 1))
obs, reward, terminated, truncated, info = env.step(blind_action)

print("\nphase:", info["phase"])
print(blind_action, "-> reward:", reward, "terminated:", terminated, "truncated:", truncated)
print("combat draw-pile deck_ranks:", obs["deck_ranks"])
print("combat draw-pile deck_suits:", obs["deck_suits"])



phase: COMBAT
45 -> reward: 0.0 terminated: False truncated: False
combat draw-pile deck_ranks: [2 3 4 3 4 4 3 3 4 3 3 4 4]
combat draw-pile deck_suits: [10 12  9 13]


In [3]:

# These are the hand arrays already exposed by the wrapper observation.
print("\nwrapper hand arrays")
print("hand_card_ids:", obs["hand_card_ids"])
print("hand_card_enhancements:", obs["hand_card_enhancements"])
print("hand_card_editions:", obs["hand_card_editions"])
print("hand_card_seals:", obs["hand_card_seals"])
print("hand_is_face_down:", obs["hand_is_face_down"])
print("hand_is_selected:", obs["hand_is_selected"])



wrapper hand arrays
hand_card_ids: [ 6 26  2 38 28  0 13 40]
hand_card_enhancements: [0 0 0 0 0 0 0 0]
hand_card_editions: [0 0 0 0 0 0 0 0]
hand_card_seals: [0 0 0 0 0 0 0 0]
hand_is_face_down: [0 0 0 0 0 0 0 0]
hand_is_selected: [0 0 0 0 0 0 0 0]


In [4]:

# Access the exact underlying state when you need full card-level data.
state = env.unwrapped.state
hand_set = set(state.hand_indexes)

print("\nbase state summary")
print("len(state.deck):", len(state.deck))
print("state.hand_indexes:", state.hand_indexes)
print("state.selected_cards:", state.selected_cards)

# Build exact hand card info from the base state.
# card_id follows: (rank - 2) * 4 + suit.
hand_cards = []
for hand_pos, deck_idx in enumerate(state.hand_indexes):
    card = state.deck[deck_idx]
    card_state = state.card_states.get(deck_idx)
    hand_cards.append({
        "hand_pos": hand_pos,
        "deck_idx": deck_idx,
        "card_id": int(card),
        "rank": int(card.rank),
        "suit": int(card.suit),
        "enhancement": int(card_state.enhancement) if card_state else 0,
        "edition": int(card_state.edition) if card_state else 0,
        "seal": int(card_state.seal) if card_state else 0,
    })

print("\nexact hand cards from base state")
for row in hand_cards:
    print(row)



base state summary
len(state.deck): 52
state.hand_indexes: [0, 1, 2, 3, 4, 5, 6, 7]
state.selected_cards: []

exact hand cards from base state
{'hand_pos': 0, 'deck_idx': 0, 'card_id': 6, 'rank': 3, 'suit': 2, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 1, 'deck_idx': 1, 'card_id': 26, 'rank': 8, 'suit': 2, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 2, 'deck_idx': 2, 'card_id': 2, 'rank': 2, 'suit': 2, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 3, 'deck_idx': 3, 'card_id': 38, 'rank': 11, 'suit': 2, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 4, 'deck_idx': 4, 'card_id': 28, 'rank': 9, 'suit': 0, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 5, 'deck_idx': 5, 'card_id': 0, 'rank': 2, 'suit': 0, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 6, 'deck_idx': 6, 'card_id': 13, 'rank': 5, 'suit': 1, 'enhancement': 0, 'edition': 0, 'seal': 0}
{'hand_pos': 7, 'deck_idx': 7, 'card_id': 40, 'rank': 12, 'suit': 0, 'enhancement

In [5]:

# The wrapper does not currently expose deck_card_* arrays.
# If you want them, build them from the remaining draw pile in the base state.
draw_pile_indexes = [deck_idx for deck_idx in range(len(state.deck)) if deck_idx not in hand_set]


# 从state.deck[] 中可以得到所有牌的信息，按照draw_pile_indexes的顺序构建对应的deck_card_*数组。
deck_card_ids = np.array([int(state.deck[deck_idx]) for deck_idx in draw_pile_indexes], dtype=np.int16)
deck_card_enhancements = np.array([
    int(state.card_states.get(deck_idx).enhancement) if state.card_states.get(deck_idx) else 0
    for deck_idx in draw_pile_indexes
], dtype=np.int8)
deck_card_editions = np.array([
    int(state.card_states.get(deck_idx).edition) if state.card_states.get(deck_idx) else 0
    for deck_idx in draw_pile_indexes
], dtype=np.int8)
deck_card_seals = np.array([
    int(state.card_states.get(deck_idx).seal) if state.card_states.get(deck_idx) else 0
    for deck_idx in draw_pile_indexes
], dtype=np.int8)

print("\nmanual deck arrays from base state (draw pile only)")
print("deck_card_ids:", deck_card_ids)
print("deck_card_enhancements:", deck_card_enhancements)
print("deck_card_editions:", deck_card_editions)
print("deck_card_seals:", deck_card_seals)

# If you only need compact distribution features for training, use the wrapper obs.
print("\ncompact histogram check")
print("draw pile size from histogram:", int(obs["deck_ranks"].sum()))
print("draw pile size from manual arrays:", len(deck_card_ids))


manual deck arrays from base state (draw pile only)
deck_card_ids: [27 37 47 48 35 36  3 49 18 41 10  7  9  8 39 34 30 33 14 25 23 15 19 42
 45 50 43 31 16 29 20 21 51 12 46 32 44 17  1 22 24  5 11  4]
deck_card_enhancements: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]
deck_card_editions: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]
deck_card_seals: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0]

compact histogram check
draw pile size from histogram: 44
draw pile size from manual arrays: 44


In [6]:
state
print(obs['action_mask'])
print("------------------------")
print(info['phase'])

[0 0 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
------------------------
COMBAT


In [8]:
# sample code for hand type info

# hand_levels rows are [id, level, chip, mult] in HandType enum order.
hand_levels = obs["hand_levels"]
high_card_id, high_card_level, high_card_chip, high_card_mult = hand_levels[0]
print("high_card [id, level, chip, mult]:", [
    int(high_card_id),
    int(high_card_level),
    int(high_card_chip),
    int(high_card_mult),
])

print("phase:", info["phase"])
print("obs keys:", sorted(obs.keys()))
print("transition deck_ranks:", obs["deck_ranks"])
print("transition deck_suits:", obs["deck_suits"])
print("next_boss_blind_id:", int(obs["next_boss_blind_id"]))

high_card [level, chip, mult]: [1, 5, 1]
phase: COMBAT
obs keys: ['action_mask', 'ante', 'blind_reward', 'blind_type', 'boss_id', 'boss_is_active', 'consumable_ids', 'consumable_is_empty', 'consumable_sell_values', 'current_score', 'deck_ranks', 'deck_suits', 'discards_remaining', 'hand_card_editions', 'hand_card_enhancements', 'hand_card_ids', 'hand_card_seals', 'hand_is_debuffed', 'hand_is_face_down', 'hand_is_selected', 'hand_levels', 'hand_size', 'hands_played_round', 'hands_remaining', 'joker_ids', 'joker_is_empty', 'joker_sell_values', 'money', 'next_boss_blind_id', 'phase', 'reroll_cost', 'round', 'shop_costs', 'shop_is_empty', 'shop_item_ids', 'shop_item_types', 'target_score', 'vouchers_owned']
transition deck_ranks: [2 3 4 3 4 4 3 3 4 3 3 4 4]
transition deck_suits: [10 12  9 13]
next_boss_blind_id: 13
